<a href="https://colab.research.google.com/github/sasya025/Travclan_A2/blob/main/SECTION_3_SQL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import pandas as pd
import sqlite3

# Load CSV
df = pd.read_csv("hotel_bookings (1).csv")

# Create SQLite database
conn = sqlite3.connect("hotel.db")

# Load CSV into SQL table
df.to_sql(
    "hotel_bookings",
    conn,
    if_exists="replace",
    index=False
)

print("Database created successfully!")

Database created successfully!


In [8]:
query = """
SELECT *
FROM hotel_bookings
LIMIT 5;
"""

pd.read_sql(query, conn)

,booking_id,customer_id,customer_name,customer_segment,customer_signup_date,customer_home_city,customer_loyalty_tier,property_id,property_name,property_city,...,nights,booking_channel,adr,discount_amount,coupon_code,total_amount,payment_method,booking_status,review_rating,review_date
0,100000,424,Customer_424,Group,2023-07-03,Manali,Platinum,38,Crimson Courtyard,Manali,...,4,OTA,5808.46,0.00,NONE,46467.68,Debit Card,Cancelled,NaN,None
1,100001,239,Customer_239,Individual,2022-07-18,Jaipur,None,32,Saffron Palace,Pune,...,4,Corporate Portal,4021.62,0.00,None,16086.48,Net Banking,Cancelled,NaN,None
2,100002,301,Customer_301,Corporate,2023-07-05,Jaipur,Gold,53,Saffron Heights,Chennai,...,3,Corporate Portal,17663.11,15373.35,SAVE10,90605.31,Net Banking,Completed,NaN,None
3,100003,722,Customer_722,Individual,2022-11-07,Udaipur,None,43,Indigo Lodge,Bangalore,...,3,OTA,5885.85,0.00,NONE,17657.55,Credit Card,No-Show,NaN,None
4,100004,306,Customer_306,Corporate,2022-02-02,Udaipur,Silver,29,Cedar Lodge,Kochi,...,7,Direct Website,6199.44,5924.82,SAVE10,80867.34,Credit Card,Completed,6.0,2024-11-21


In [9]:
import sqlite3
print(sqlite3.sqlite_version)

3.37.2


In [10]:
query = """
WITH room_counts AS (
    SELECT
        property_type,
        room_type,
        COUNT(*) AS booking_count,
        DENSE_RANK() OVER (
            PARTITION BY property_type
            ORDER BY COUNT(*) DESC
        ) AS dr
    FROM hotel_bookings
    WHERE booking_status = 'Completed'
    GROUP BY property_type, room_type
)

SELECT
    property_type,
    room_type,
    booking_count
FROM room_counts
WHERE dr = 1;
"""

result = pd.read_sql(query, conn)
result

,property_type,room_type,booking_count
0,Budget,Standard,1480
1,Luxury,Standard,538
2,Mid-Range,Standard,1792
3,Premium,Standard,814


In [11]:
query = """
WITH monthly_revenue AS (
    SELECT
        substr(checkin_date,1,7) AS month,
        SUM(CAST(total_amount AS REAL)) AS monthly_revenue
    FROM hotel_bookings
    WHERE booking_status = 'Completed'
    GROUP BY month
)

SELECT
    month,
    monthly_revenue,
    SUM(monthly_revenue) OVER (
        ORDER BY month
    ) AS cumulative_revenue
FROM monthly_revenue
ORDER BY month;
"""

result = pd.read_sql(query, conn)
result

,month,monthly_revenue,cumulative_revenue
0,2024-01,27854101.82,2.785410e+07
1,2024-02,23589488.55,5.144359e+07
2,2024-03,21996699.31,7.344029e+07
3,2024-04,23311117.49,9.675141e+07
4,2024-05,24588064.29,1.213395e+08
5,2024-06,21613127.57,1.429526e+08
6,2024-07,19615670.65,1.625683e+08
7,2024-08,22556399.71,1.851247e+08
8,2024-09,24394077.58,2.095187e+08
9,2024-10,27170147.80,2.366889e+08
